In [51]:
import pickle
from pathlib import Path

import geopandas as gpd
import numpy as np
import osmnx as ox
import polars as pl
from pyproj import Transformer
from valhalla import Actor, get_config

transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

ox.settings.log_console = False
ox.settings.use_cache = True

In [2]:
# def read_street_edge_list(device_group: int, path: str) -> list:
#     print(f"{device_group:02x}")
#     try:
#         with open(path, "br") as fp:
#             street_edge_list = pickle.load(fp)
#     except FileNotFoundError:
#         print(f"{path} was not found, generating it...")
#         df = prepare_parquet(device_group, days=opts.days, dir=opts.dir)
#         # runs more than 15 min on a full partition
#         street_edge_list = ox.distance.nearest_edges(
#             g_, X=df.geometry.x, Y=df.geometry.y
#         )  # , interpolate=25)
#         with open(path, "bw") as fp:
#             pickle.dump(street_edge_list, fp)
#     return street_edge_list

In [2]:
config = get_config(
    tile_extract="../data/valhalla/hungary/valhalla_tiles.tar",
    tile_dir="../data/valhalla/hungary/",
)
actor = Actor(config)

In [50]:
DIR = "/mnt/common-ssd/group-dirs/tkom-2026-project/pickwell/budapest/partitioned"

In [ ]:
for i in range(1):
    df = pl.read_parquet(f"{DIR}/with_speed/{i:02x}.parquet")

In [6]:
# actor.trace_route()

In [7]:
# request = {
#     "shape": [
#         {"lat": 47.47992, "lon": 19.074155, "type": "break"},
#         {"lat": 47.479995, "lon": 19.074221, "type": "via"},
#         {"lat": 47.476752, "lon": 19.075154, "type": "break"},
#     ],
#     "costing": "auto",
#     # "shape_match": "edge_walk",
# }

# result = actor.trace_route(request)

In [8]:
# request = {
#     "shape": [
#         {"lat": 47.47992, "lon": 19.074155, "type": "break"},
#         {"lat": 47.479995, "lon": 19.074221, "type": "via"},
#         {"lat": 47.476752, "lon": 19.075154, "type": "break"},
#     ],
#     "costing": "auto",
#     # "shape_match": "edge_walk",
# }

# result = actor.trace_attributes(request)

In [5]:
area = gpd.read_file("../data/osm/budapest/boundary.geojson")

In [ ]:
# try:
#     g_drive = ox.load_graphml("../output/budapest/drive.graphml")
# except FileNotFoundError:
#     # 2m 1s
#     g_drive = ox.graph_from_polygon(
#         area.geometry[0], network_type="drive", simplify=False
#     )

#     ox.save_graphml(g_drive, "../output/budapest/drive.graphml")

In [7]:
try:
    g_all = ox.load_graphml("../output/budapest/all.graphml")
except FileNotFoundError:
    # 4m
    g_all = ox.graph_from_polygon(area.geometry[0], network_type="all", simplify=False)
    ox.save_graphml(g_all, "../output/budapest/all.graphml")

In [8]:
g_all_proj = ox.project_graph(g_all, to_crs=3857)

In [31]:
xs, ys = transformer.transform(
    df["longitude"].to_numpy(),
    df["latitude"].to_numpy(),
)

df = df.with_columns(
    pl.Series("lon", xs),
    pl.Series("lat", ys),
)

In [ ]:
try:
    with open("../output/budapest/street_edge_list.pickle", "rb") as fp:
        street_edge_list = pickle.load(fp)
except FileExistsError:
    street_edge_list = ox.distance.nearest_edges(
        g_all_proj,
        X=df["lon"],
        Y=df["lat"],
        return_dist=True,
    )
    # 7m 45s
    with open("../output/budapest/street_edge_list.pickle", "wb") as fp:
        pickle.dump(street_edge_list, fp)

In [17]:
len(street_edge_list[0])

15460994

In [32]:
len(df)

15460994

In [22]:
osmids = [g_all.edges[u, v, key]["osmid"] for u, v, key in street_edge_list[0]]

In [ ]:
df = df.with_columns(
    pl.Series("street", osmids),
    pl.Series("distance_from_street", np.round(street_edge_list[1]).astype(int)),
)

In [52]:
Path(f"{DIR}/with_street/").mkdir(exist_ok=True, parents=True)

In [53]:
df.write_parquet(f"{DIR}/with_street/00.parquet")